# Exploring the annual report extraction

A four-step walkthrough of `extract_annual_report.py`, the script that turns the
Stora Enso Annual Report 2025 PDF into tidy tables for Python and Power BI.

**Before running:** put these three files in the same folder as this notebook.

| File | Where it comes from |
|---|---|
| `extract_annual_report.py` | this repository |
| `STORAENSO_Annual_Report_2025.pdf` | storaenso.com/investors |
| this notebook | this repository |

Then install the libraries (once):

```
pip install pdfplumber pandas openpyxl matplotlib jupyter
```

Run a cell with **Shift+Enter**.

## Setup

`pdfplumber` reads text and word positions out of a PDF. The helper functions come
from the script itself, so the notebook and the script always behave the same way.

In [ ]:
from pathlib import Path

import pdfplumber
import pandas as pd

from extract_annual_report import (
    column_text,        # crop one column of a page and return its text
    parse_table_lines,  # keep lines that look like "label number number"
    to_number,          # '7.8%' -> 0.078
    SEGMENT_COLUMNS,    # the column positions used for pages 53-54
)

PDF_PATH = Path("STORAENSO_Annual_Report_2025.pdf")
assert PDF_PATH.exists(), f"Put the report PDF next to this notebook (looked for {PDF_PATH})"

pdf = pdfplumber.open(PDF_PATH)
print(f"{len(pdf.pages)} pages loaded")
print("Column crops used for the segment pages:", SEGMENT_COLUMNS)

## Cell 1 — What one column actually looks like

Page 53 holds three segment tables side by side. Reading the whole page reads straight
across all three, mixing them together. Cropping one column first gives a clean table.

Run the cell, then change `(370, 700)` to `(40, 370)` for Packaging Materials or
`(700, None)` for Biomaterials.

In [ ]:
page = pdf.pages[52]          # page 53 — Python counts pages from 0
x0, x1 = 370, 700             # middle column: Packaging Solutions

text = column_text(page, x0, x1)
print(text[:900])

For comparison, this is the same page read without cropping. Notice how the numbers
of three different segments end up on one line.

In [ ]:
whole_page = page.extract_text()
for line in whole_page.split("\n"):
    if line.strip().startswith("Sales"):
        print(line)

## Cell 2 — What the parser keeps

`parse_table_lines(text, 2)` keeps only lines that end with exactly two numbers
(2025 and 2024) and converts them: `1,027` becomes `1027.0`, `7.8%` becomes `0.078`.

Watch the last row: the label *Corrugated packaging European deliveries, million m2*
is split over two lines in the PDF and is rejoined automatically.

In [ ]:
for label, values in parse_table_lines(text, 2):
    print(f"{label:<50} {values}")

Why percentages become decimals: Power BI (and pandas) expect `0.078` and format it
as `7.8%`. Storing `7.8` would display as `780%`.

In [ ]:
for sample in ["1,027", "-15", "7.8%", "2.8"]:
    print(f"{sample:>8}  ->  {to_number(sample)}")

## Cell 3 — Finding column positions in any PDF

This is how the numbers in the CONFIG block were found, and how you would adapt the
script to another company's report.

The output lists where each occurrence of "Sales" or "EUR" starts horizontally (`x0`).
Clusters of similar x values mark the columns; set each crop boundary just to the left
of a cluster.

In [ ]:
from collections import Counter

words = pdf.pages[52].extract_words()

# Every table on these pages starts with a "Sales" row, so the left edges of
# the word "Sales" mark where each column begins.
edges = Counter(round(w["x0"]) for w in words if w["text"] == "Sales")
print("Left edges of 'Sales':", sorted(edges))

for w in words:
    if w["text"] == "Sales":
        print(f"x0={round(w['x0']):>4}  top={round(w['top']):>4}")

The table columns start at 45, 375 and 705, which is why the crops are
`(40, 370)`, `(370, 700)` and `(700, None)` — each boundary sits just to the left of a
column. The stray 547 is the word "Sales" inside a paragraph, not a table, which is why
it helps to look at the whole list rather than assume.

Try it on another page: change `pdf.pages[52]` to `pdf.pages[59]` (page 60, the
sensitivity table) and see where its columns start.

## Cell 4 — Working with the results in pandas

Run the script first so the `output/` folder exists:

```
python extract_annual_report.py --pdf STORAENSO_Annual_Report_2025.pdf
```

The long format has one row per segment, year and metric — easy to filter and pivot.

In [ ]:
df = pd.read_csv("output/segments_long.csv")
print(df.shape, "rows, columns")
df.head()

In [ ]:
# Everything reported for Packaging Solutions in 2025
ps = df[(df.segment == "Packaging Solutions") & (df.year == 2025)]
ps[["label", "value"]].to_string(index=False)

In [ ]:
# Margin by segment, 2025 vs 2024 — a pivot, like in Excel
margin = (df[df.metric == "adj_ebit_margin"]
          .pivot(index="segment", columns="year", values="value")
          .sort_values(2025, ascending=False))
(margin * 100).round(1)

In [ ]:
# Year-on-year EBIT change by segment
ebit = df[df.metric == "adj_ebit"].pivot(index="segment", columns="year", values="value")
ebit["change"] = ebit[2025] - ebit[2024]
ebit.sort_values("change", ascending=False)

In [ ]:
# A quick chart: adjusted EBIT by segment
ax = ebit[[2024, 2025]].plot(kind="bar", figsize=(9, 4),
                             title="Adjusted EBIT by segment (EUR million)")
ax.set_xlabel("")
ax.axhline(0, linewidth=0.8, color="black")

## Practice exercises

| # | Exercise | Skill |
|---|---|---|
| 1 | Add `cash_flow_operations` to `fact_segment` (edit the metric list in `build_fact_segment`) | pivoting |
| 2 | Crop the wrong column on purpose, e.g. `(300, 700)`, and see which check in `validate()` catches it | why validation matters |
| 3 | Extract the cost table (Table 2, page 60) into a `fact_cost_mix` DataFrame | writing your own extractor |
| 4 | Run the script on a quarterly interim report and adjust the CONFIG | reusability |

Starting point for exercise 3 — the values are percentages, and the table sits below
"Table 2" in the left column of page 60:

In [ ]:
cost_text = column_text(pdf.pages[59], 40, 530).split("Table 2")[1]
for label, values in parse_table_lines(cost_text, 2):
    print(f"{label:<32} {values}")

Turn that into a DataFrame with columns `cost_item`, `share_of_costs`, `share_of_sales`,
and check the shares of costs add up to about 100%.